In [ ]:
from aiida.engine import run
from aiida.orm import Dict, KpointsData, StructureData, load_code, load_group

In [27]:
from aiida import load_profile
load_profile('bizet')

Profile<uuid='4dffa2c17bb4476a84bb6d97e33aacb1' name='puccini'>

In [28]:
# Load the code configured for ``pw.x``. Make sure to replace this string
# with the label that you used in the setup of the code.
code = load_code('qe-6.7-pw@localhost')
builder = code.get_builder()

In [29]:
# Create a silicon fcc crystal
from ase.build import bulk
structure = StructureData(ase=bulk('Si', 'fcc', 5.43))
builder.structure = structure

In [30]:
# Load the pseudopotential family.
pseudo_family = load_group('SSSP/1.3/PBE/efficiency')
builder.pseudos = pseudo_family.get_pseudos(structure=structure)

In [31]:
# Request the recommended wavefunction and charge density cutoffs
# for the given structure and energy units.
cutoff_wfc, cutoff_rho = pseudo_family.get_recommended_cutoffs(
    structure=structure,
    unit='Ry'
)

parameters = Dict({
    'CONTROL': {
        'calculation': 'scf'
    },
    'SYSTEM': {
        'ecutwfc': cutoff_wfc,
        'ecutrho': cutoff_rho,
    }
})
builder.parameters = parameters

In [32]:
# Generate a 2x2x2 Monkhorst-Pack mesh
kpoints = KpointsData()
kpoints.set_kpoints_mesh([2, 2, 2])
builder.kpoints = kpoints

In [33]:
# Run the calculation on 1 CPU and kill it if it runs longer than 1800 seconds.
# Set ``withmpi`` to ``False`` if ``pw.x`` was compiled without MPI support.
builder.metadata.options = {
    'resources': {
        'num_machines': 1,
    },
    'max_wallclock_seconds': 1800,
    'withmpi': False,
}

All required inputs have now been defined. The calculation can be started by launching the process builder:

In [34]:
results, node = run.get_node(builder)

ConnectionError: [Errno 111] Connect call failed ('127.0.0.1', 5672)

When the calculation is finished, run.get_node will return a tuple of two values: results and node. The results will be a dictionary of all the output nodes that were produced and the node will hold a reference to the node that represents the calculation in AiiDA’s provenance graph. When the calculation is finished, check whether it was successful through its exit status:

node.exit_status